# Chapter 3.1: vLLM Codebase Tour

## Learning Objectives

By the end of this notebook, you will:

1. Understand the overall structure of the vLLM repository
2. Know the purpose of each major directory and key files
3. Understand the build system (setup.py, CMake for CUDA extensions)
4. Know the main entry points for vLLM (CLI, API server, Python API)
5. Understand the configuration system (EngineArgs, ModelConfig, etc.)
6. Be able to trace the dependency graph between major components
7. Navigate the codebase effectively to find specific functionality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.1_codebase_tour.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.1_codebase_tour.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Repository Structure Overview

The vLLM repository is a large, well-organized codebase. Let's start with the high-level directory tree to understand how it is laid out.

```
vllm/                          # Root of the repository
├── vllm/                      # Main Python package
│   ├── engine/                # Core engine: LLMEngine, AsyncLLMEngine
│   ├── worker/                # GPU/CPU workers that execute model inference
│   ├── model_executor/        # Model loading, execution, and layer definitions
│   ├── attention/             # Attention backends (FlashAttention, FlashInfer, etc.)
│   ├── core/                  # Scheduler, block manager, memory management
│   ├── entrypoints/           # API servers, CLI, Python-level entry points
│   ├── lora/                  # LoRA adapter support
│   ├── spec_decode/           # Speculative decoding
│   ├── distributed/           # Tensor/pipeline parallelism communication
│   ├── transformers_utils/    # HuggingFace transformers integration utilities
│   ├── executor/              # Executors that manage workers (GPU, CPU, Ray, etc.)
│   └── ...
├── csrc/                      # C++/CUDA source code for custom kernels
│   ├── attention/             # PagedAttention CUDA kernels
│   ├── quantization/          # Quantization kernels (GPTQ, AWQ, etc.)
│   └── ...
├── tests/                     # Test suite
├── benchmarks/                # Benchmarking scripts
├── docs/                      # Documentation
├── examples/                  # Example usage scripts
├── setup.py                   # Package build configuration
├── CMakeLists.txt             # CMake build for CUDA extensions
└── pyproject.toml             # Modern Python project metadata
```

## 2. Key Directory Deep Dive

Let's examine each major directory in detail.

### 2.1 `vllm/engine/` — The Heart of vLLM

```
vllm/engine/
├── __init__.py
├── llm_engine.py          # LLMEngine: synchronous core engine
├── async_llm_engine.py    # AsyncLLMEngine: async wrapper for serving
├── arg_utils.py           # EngineArgs, AsyncEngineArgs
├── output.py              # RequestOutput, CompletionOutput
└── metrics.py             # Prometheus metrics for engine monitoring
```

**Key Insight**: `LLMEngine` is the synchronous core that does all the heavy lifting. `AsyncLLMEngine` wraps it with asyncio for high-throughput serving. Almost every operation flows through `LLMEngine.step()`.

### 2.2 `vllm/core/` — Scheduler and Memory Management

```
vllm/core/
├── __init__.py
├── scheduler.py           # Scheduler: decides which sequences run next
├── block_manager.py       # BlockSpaceManager: manages KV cache blocks  
│                          # (v1 implementation)
├── block_manager_v2.py    # BlockSpaceManagerV2: improved block manager
├── block/                 # Modular block allocation components
│   ├── block_table.py     # Block table management
│   ├── cpu_gpu_block_allocator.py
│   ├── naive_block.py     # Simple block allocator
│   └── prefix_caching_block.py  # Prefix-caching aware allocator
├── evictor.py             # Block eviction policies
└── interfaces.py          # Abstract interfaces for block management
```

**Key Insight**: The scheduler and block manager work together. The scheduler decides *which* sequences to run; the block manager decides *where* their KV cache blocks live in GPU memory.

### 2.3 `vllm/worker/` — GPU Execution

```
vllm/worker/
├── __init__.py
├── worker.py              # Worker: per-GPU process managing execution
├── model_runner.py        # ModelRunner: builds inputs, runs forward pass
├── cache_engine.py        # CacheEngine: manages KV cache tensors on GPU
├── worker_base.py         # Abstract base for workers
├── cpu_worker.py          # CPU-only worker
├── neuron_worker.py       # AWS Neuron worker
├── tpu_worker.py          # TPU worker (experimental)
└── ...
```

**Key Insight**: Each GPU gets one Worker. The Worker owns a ModelRunner (which does the actual forward pass) and a CacheEngine (which manages the physical KV cache memory).

### 2.4 `vllm/model_executor/` — Model Definitions and Loading

```
vllm/model_executor/
├── __init__.py
├── model_loader/          # Model weight loading logic
│   ├── loader.py          # Main loader implementations
│   ├── weight_utils.py    # Weight download, conversion utilities
│   └── ...
├── models/                # Model architecture implementations
│   ├── llama.py           # LLaMA family
│   ├── gpt2.py            # GPT-2
│   ├── mistral.py         # Mistral
│   ├── qwen2.py           # Qwen2
│   ├── registry.py        # ModelRegistry: maps model names → classes
│   └── ... (50+ model files)
├── layers/                # Custom layer implementations
│   ├── linear.py          # TP-aware linear layers
│   ├── attention.py       # Attention layer wrapper
│   ├── rotary_embedding.py # RoPE implementations
│   ├── quantization/      # Quantization method implementations
│   │   ├── gptq.py
│   │   ├── awq.py
│   │   └── ...
│   └── ...
├── sampling_metadata.py   # Metadata for the sampler
└── ...
```

**Key Insight**: vLLM re-implements every model architecture from scratch (rather than using HuggingFace's implementations) to integrate PagedAttention and tensor parallelism directly into the model layers.

### 2.5 `vllm/attention/` — Attention Backends

```
vllm/attention/
├── __init__.py
├── attention.py           # Attention class: the unified interface
├── backends/              # Backend implementations
│   ├── abstract.py        # AttentionBackend abstract base
│   ├── flash_attn.py      # FlashAttention v2 backend
│   ├── flashinfer.py      # FlashInfer backend
│   ├── xformers.py        # xFormers backend
│   ├── torch_sdpa.py      # PyTorch native SDPA
│   └── ...
├── ops/                   # Attention-related operations
│   └── paged_attn.py      # PagedAttention operation wrappers
└── selector.py            # Backend selection logic
```

**Key Insight**: The attention module provides a **pluggable backend system**. Different hardware and configurations benefit from different attention implementations. The `selector.py` auto-detects the best backend.

### 2.6 `vllm/entrypoints/` — How Users Interact with vLLM

```
vllm/entrypoints/
├── __init__.py
├── llm.py                 # LLM class: high-level Python API (offline inference)
├── openai/                # OpenAI-compatible API server
│   ├── api_server.py      # FastAPI application setup
│   ├── serving_chat.py    # /v1/chat/completions endpoint
│   ├── serving_completion.py  # /v1/completions endpoint
│   ├── serving_models.py  # /v1/models endpoint
│   ├── protocol.py        # Request/response Pydantic models
│   └── ...
├── chat_utils.py          # Chat template handling
└── ...
```

## 3. Build System

vLLM uses a hybrid build system because it has both Python code and C++/CUDA extensions.

In [ ]:
# Let's examine how the build system is structured
# The key files involved in building vLLM:

build_files = {
    "setup.py": """
    The main setup script. Key responsibilities:
    1. Detects CUDA availability and version
    2. Invokes CMake to build C++/CUDA extensions
    3. Sets up the Python package with dependencies
    
    Key environment variables:
    - VLLM_TARGET_DEVICE: 'cuda', 'rocm', 'cpu', etc.
    - MAX_JOBS: parallel compilation jobs
    - NVCC_THREADS: CUDA compiler threads
    """,
    
    "CMakeLists.txt": """
    CMake configuration for C++/CUDA extensions.
    Builds:
    - PagedAttention CUDA kernels
    - Quantization kernels (GPTQ, AWQ, SqueezeLLM)
    - Activation kernels (SiLU, GELU)
    - Layernorm kernels
    - Sampling kernels
    - Cache management operations
    """,
    
    "pyproject.toml": """
    Modern Python project metadata.
    Defines:
    - Build system requirements (setuptools, cmake, ninja, torch)
    - Project metadata
    - Optional dependencies
    """
}

for filename, description in build_files.items():
    print(f"=== {filename} ===")
    print(description)

In [ ]:
# setup.py key sections (annotated pseudocode)

setup_py_walkthrough = """
# === setup.py Key Sections ===

# 1. Device Detection
VLLM_TARGET_DEVICE = os.environ.get("VLLM_TARGET_DEVICE", "cuda")
# This determines which C++ extensions to build

# 2. CMake Extension Class
class cmake_build_ext(build_ext):
    def build_extensions(self):
        # Configure CMake with proper CUDA architectures
        # Build all C++ extensions in one CMake invocation
        # This is more efficient than building each extension separately
        cmake_args = [
            '-DVLLM_TARGET_DEVICE=' + VLLM_TARGET_DEVICE,
            '-DCMAKE_BUILD_TYPE=Release',
            # GPU architecture flags (sm_70, sm_80, sm_89, sm_90)
        ]
        subprocess.check_call(['cmake', '..'] + cmake_args)
        subprocess.check_call(['cmake', '--build', '.'])

# 3. Extension Modules
ext_modules = [
    CMakeExtension(name="vllm._C"),  # Main C++ extension
]

# 4. Package Setup
setup(
    name="vllm",
    ext_modules=ext_modules,
    cmdclass={"build_ext": cmake_build_ext},
    install_requires=[...],  # Runtime dependencies
)
"""
print(setup_py_walkthrough)

### 3.1 CUDA Kernels in `csrc/`

```
csrc/
├── attention/
│   ├── attention_kernels.cu    # PagedAttention v1 & v2 CUDA kernels
│   ├── attention_utils.cuh    # Shared attention utilities
│   └── ...
├── quantization/
│   ├── gptq/                  # GPTQ dequantization kernels
│   ├── awq/                   # AWQ kernels
│   └── ...
├── activation_kernels.cu      # Fused activation functions
├── layernorm_kernels.cu       # Fused layer normalization
├── cache_kernels.cu           # KV cache copy/swap operations
├── pos_encoding_kernels.cu    # Rotary position embedding
├── dispatch_utils.h           # CUDA dispatch helpers
├── torch_bindings.cpp         # PyTorch C++ extension bindings
└── ...
```

The `torch_bindings.cpp` file registers all CUDA operations as PyTorch custom ops, making them callable from Python via `vllm._C` or `torch.ops.vllm`.

## 4. Entry Points

vLLM provides multiple ways to start and use the system:

In [ ]:
# Entry Point 1: CLI command `vllm serve`
# Defined in setup.py/pyproject.toml as a console_scripts entry point

cli_entry = """
# In pyproject.toml:
[project.scripts]
vllm = "vllm.scripts:main"

# vllm/scripts.py:
def main():
    parser = argparse.ArgumentParser()
    subparsers = parser.add_subparsers()
    
    # 'vllm serve' subcommand
    serve_parser = subparsers.add_parser('serve')
    serve_parser.set_defaults(func=serve)  # calls the serve function
    
    # The serve function:
    def serve(args):
        # Ultimately calls:
        #   uvicorn.run("vllm.entrypoints.openai.api_server:app", ...)
        # This starts a FastAPI server with OpenAI-compatible endpoints

Usage:
  vllm serve meta-llama/Llama-3-8B-Instruct \\
    --tensor-parallel-size 2 \\
    --max-model-len 4096
"""
print(cli_entry)

In [ ]:
# Entry Point 2: Python module execution
# python -m vllm.entrypoints.openai.api_server

api_server_entry = """
# vllm/entrypoints/openai/api_server.py:

# 1. Parse arguments (model, TP size, max tokens, etc.)
# 2. Create AsyncLLMEngine from EngineArgs
# 3. Set up FastAPI app with routes:
#    - POST /v1/chat/completions  → chat_completion()
#    - POST /v1/completions       → completion()
#    - GET  /v1/models            → show_available_models()
#    - GET  /health               → health_check()
# 4. Start uvicorn server

if __name__ == "__main__":
    # This is the entry point when running:
    # python -m vllm.entrypoints.openai.api_server --model <model>
    args = parse_args()
    engine = AsyncLLMEngine.from_engine_args(EngineArgs(**vars(args)))
    app = build_app(engine)
    uvicorn.run(app, host=args.host, port=args.port)
"""
print(api_server_entry)

In [ ]:
# Entry Point 3: Python API (offline batch inference)
# This is the simplest way to use vLLM programmatically

python_api_entry = """
# vllm/entrypoints/llm.py:

class LLM:
    '''High-level Python API for offline inference.'''
    
    def __init__(self, model: str, **kwargs):
        # Creates EngineArgs from user parameters
        engine_args = EngineArgs(model=model, **kwargs)
        # Creates LLMEngine (synchronous)
        self.llm_engine = LLMEngine.from_engine_args(engine_args)
    
    def generate(self, prompts, sampling_params, ...):
        # 1. Add all requests to the engine
        for prompt in prompts:
            self.llm_engine.add_request(request_id, prompt, sampling_params)
        
        # 2. Run engine loop until all requests complete
        while self.llm_engine.has_unfinished_requests():
            step_outputs = self.llm_engine.step()
            # Collect finished outputs
        
        return outputs

# Usage:
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Llama-3-8B-Instruct")
outputs = llm.generate(["Hello, world!"], SamplingParams(temperature=0.8))
"""
print(python_api_entry)

## 5. Configuration System

vLLM has a rich configuration system organized into multiple config dataclasses. Understanding these is critical because they control every aspect of the engine's behavior.

In [ ]:
# The configuration hierarchy

config_hierarchy = """
EngineArgs (user-facing)
  │
  ├── Creates: ModelConfig
  │     - model: str                    # HuggingFace model name/path
  │     - tokenizer: str                # Tokenizer name/path
  │     - dtype: str                    # 'auto', 'float16', 'bfloat16'
  │     - max_model_len: int            # Maximum sequence length
  │     - quantization: str             # 'gptq', 'awq', 'squeezellm', None
  │     - enforce_eager: bool           # Disable CUDA graphs
  │     - trust_remote_code: bool       # Allow custom model code
  │     - seed: int                     # Random seed
  │     - revision: str                 # Model revision/commit
  │
  ├── Creates: CacheConfig
  │     - block_size: int               # Tokens per KV cache block (default: 16)
  │     - gpu_memory_utilization: float  # Fraction of GPU memory for KV cache (default: 0.90)
  │     - swap_space: int               # CPU swap space in GiB
  │     - cache_dtype: str              # KV cache data type
  │     - num_gpu_blocks: int           # Computed at runtime
  │     - num_cpu_blocks: int           # Computed at runtime
  │
  ├── Creates: ParallelConfig
  │     - tensor_parallel_size: int     # Number of GPUs for TP
  │     - pipeline_parallel_size: int   # Number of stages for PP
  │     - worker_use_ray: bool          # Use Ray for distributed
  │
  ├── Creates: SchedulerConfig
  │     - max_num_batched_tokens: int   # Max tokens per iteration
  │     - max_num_seqs: int             # Max sequences per iteration
  │     - max_paddings: int             # Max padding tokens
  │     - delay_factor: float           # Scheduling delay
  │     - enable_chunked_prefill: bool  # Chunked prefill support
  │
  ├── Creates: DeviceConfig
  │     - device: str                   # 'cuda', 'cpu', 'neuron', 'tpu'
  │
  ├── Creates: LoRAConfig (optional)
  │     - max_lora_rank: int
  │     - max_loras: int
  │     - lora_extra_vocab_size: int
  │
  └── Creates: SpeculativeConfig (optional)
        - draft_model: str
        - num_speculative_tokens: int
"""
print(config_hierarchy)

In [ ]:
# Let's look at how EngineArgs creates all these configs
# Source: vllm/engine/arg_utils.py

engine_args_code = '''
@dataclass
class EngineArgs:
    """Arguments for vLLM engine."""
    model: str
    tokenizer: Optional[str] = None
    tokenizer_mode: str = 'auto'
    trust_remote_code: bool = False
    dtype: str = 'auto'
    max_model_len: Optional[int] = None
    
    # Worker/parallelism
    worker_use_ray: bool = False
    tensor_parallel_size: int = 1
    pipeline_parallel_size: int = 1
    
    # Scheduler
    max_num_batched_tokens: Optional[int] = None
    max_num_seqs: int = 256
    
    # KV Cache
    block_size: int = 16
    gpu_memory_utilization: float = 0.90
    swap_space: int = 4  # GiB
    
    # Quantization
    quantization: Optional[str] = None
    enforce_eager: bool = False
    
    def create_engine_config(self) -> EngineConfig:
        """Creates all sub-configs from these arguments."""
        model_config = ModelConfig(
            model=self.model,
            tokenizer=self.tokenizer or self.model,
            dtype=self.dtype,
            max_model_len=self.max_model_len,
            quantization=self.quantization,
            ...
        )
        cache_config = CacheConfig(
            block_size=self.block_size,
            gpu_memory_utilization=self.gpu_memory_utilization,
            swap_space=self.swap_space,
            ...
        )
        parallel_config = ParallelConfig(
            tensor_parallel_size=self.tensor_parallel_size,
            pipeline_parallel_size=self.pipeline_parallel_size,
            ...
        )
        scheduler_config = SchedulerConfig(
            max_num_batched_tokens=self.max_num_batched_tokens,
            max_num_seqs=self.max_num_seqs,
            ...
        )
        return EngineConfig(
            model_config=model_config,
            cache_config=cache_config,
            parallel_config=parallel_config,
            scheduler_config=scheduler_config,
            ...
        )
'''
print(engine_args_code)

## 6. Dependency Graph Between Major Components

Understanding how components depend on each other is crucial for navigating the codebase.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# ── vLLM Component Dependency Graph ──
# Box-and-arrow diagram: API Server → LLMEngine → Scheduler → Worker → ModelRunner → AttentionBackend
# with BlockManager connected to Scheduler

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
fig.patch.set_facecolor('white')

# Color scheme
BLUE = '#4A90D9'    # Compute
GREEN = '#27AE60'   # Memory / cache
ORANGE = '#F39C12'  # Communication / entry
RED = '#E74C3C'     # Bottleneck
PURPLE = '#8E44AD'  # I/O

# Component positions and colors: (x, y, color, width)
components = {
    'API Server\n(FastAPI)':        (4,  9.0, ORANGE, 3.0),
    'LLM\n(Python API)':           (10, 9.0, ORANGE, 2.6),
    'AsyncLLMEngine':              (7,  7.5, BLUE,   3.0),
    'LLMEngine':                   (7,  6.0, BLUE,   3.0),
    'Scheduler':                   (3.5,4.2, BLUE,   2.6),
    'BlockManager':                (3.5,2.4, GREEN,  2.6),
    'Worker':                      (10, 4.2, BLUE,   2.6),
    'ModelRunner':                 (10, 2.4, PURPLE, 2.6),
    'AttentionBackend':            (7,  0.8, RED,    3.0),
}

box_h = 0.8

for name, (x, y, color, w) in components.items():
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - box_h/2), w, box_h,
        boxstyle="round,pad=0.12", facecolor=color,
        edgecolor='white', linewidth=2, alpha=0.92)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white',
            path_effects=[pe.withStroke(linewidth=0.5, foreground='black')])

# Arrows (src → dst)
arrow_kw = dict(arrowstyle='->', color='#2C3E50', lw=1.8,
                connectionstyle='arc3,rad=0.0')

edges = [
    ('API Server\n(FastAPI)',   'AsyncLLMEngine'),
    ('LLM\n(Python API)',       'LLMEngine'),
    ('AsyncLLMEngine',          'LLMEngine'),
    ('LLMEngine',               'Scheduler'),
    ('LLMEngine',               'Worker'),
    ('Scheduler',               'BlockManager'),
    ('Worker',                   'ModelRunner'),
    ('ModelRunner',              'AttentionBackend'),
]

for src, dst in edges:
    sx, sy = components[src][0], components[src][1]
    dx, dy = components[dst][0], components[dst][1]
    ax.annotate('', xy=(dx, dy + box_h/2 + 0.05),
                xytext=(sx, sy - box_h/2 - 0.05),
                arrowprops=arrow_kw)

# Legend
legend_items = [
    mpatches.Patch(color=BLUE,   label='Compute'),
    mpatches.Patch(color=GREEN,  label='Memory / Cache'),
    mpatches.Patch(color=ORANGE, label='Entry Points'),
    mpatches.Patch(color=RED,    label='Bottleneck / Kernel'),
    mpatches.Patch(color=PURPLE, label='I/O / Model'),
]
ax.legend(handles=legend_items, loc='lower right', fontsize=9,
          framealpha=0.9, edgecolor='gray')

ax.set_title('vLLM Component Dependency Graph',
             fontsize=16, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('vLLM Component Dependency Graph', fontsize=18, fontweight='bold', pad=20)

# Define component boxes
components = {
    'API Server': (7, 11, '#FFB3BA'),
    'LLM (Python API)': (2, 11, '#FFB3BA'),
    'AsyncLLMEngine': (7, 9.5, '#BAFFC9'),
    'LLMEngine': (5, 8, '#BAFFC9'),
    'Scheduler': (2.5, 6, '#BAE1FF'),
    'BlockManager': (2.5, 4, '#BAE1FF'),
    'Executor': (8, 6, '#FFFFBA'),
    'Worker': (8, 4, '#FFFFBA'),
    'ModelRunner': (11, 4, '#FFFFBA'),
    'Model': (11, 2, '#E8BAFF'),
    'Attention Backend': (8, 2, '#E8BAFF'),
    'CacheEngine': (5, 2, '#FFD9BA'),
    'Sampler': (14, 4, '#E8BAFF'),
}

box_width = 2.8
box_height = 0.9

for name, (x, y, color) in components.items():
    rect = mpatches.FancyBboxPatch((x - box_width/2, y - box_height/2), 
                                    box_width, box_height,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center', fontsize=9, fontweight='bold')

# Draw arrows showing dependencies
arrows = [
    ('API Server', 'AsyncLLMEngine'),
    ('LLM (Python API)', 'LLMEngine'),
    ('AsyncLLMEngine', 'LLMEngine'),
    ('LLMEngine', 'Scheduler'),
    ('LLMEngine', 'Executor'),
    ('Scheduler', 'BlockManager'),
    ('Executor', 'Worker'),
    ('Worker', 'ModelRunner'),
    ('Worker', 'CacheEngine'),
    ('ModelRunner', 'Model'),
    ('ModelRunner', 'Attention Backend'),
    ('ModelRunner', 'Sampler'),
]

for src, dst in arrows:
    sx, sy, _ = components[src]
    dx, dy, _ = components[dst]
    ax.annotate('', xy=(dx, dy + box_height/2), xytext=(sx, sy - box_height/2),
                arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))

# Legend
legend_items = [
    mpatches.Patch(color='#FFB3BA', label='Entry Points'),
    mpatches.Patch(color='#BAFFC9', label='Engine'),
    mpatches.Patch(color='#BAE1FF', label='Scheduling & Memory'),
    mpatches.Patch(color='#FFFFBA', label='Execution'),
    mpatches.Patch(color='#E8BAFF', label='Model & Attention'),
    mpatches.Patch(color='#FFD9BA', label='Cache'),
]
ax.legend(handles=legend_items, loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

## 7. The Request Flow — End-to-End

Let's trace what happens when a user sends a request to vLLM, from entry to output.

In [ ]:
request_flow = """
REQUEST FLOW: User sends POST /v1/chat/completions
═══════════════════════════════════════════════════

1. API Server (vllm/entrypoints/openai/serving_chat.py)
   ├── Parse request → ChatCompletionRequest
   ├── Apply chat template → convert messages to prompt string
   ├── Tokenize prompt → token IDs
   └── Call engine.generate(prompt, sampling_params, request_id)

2. AsyncLLMEngine (vllm/engine/async_llm_engine.py)
   ├── Add request to internal queue
   └── Background loop calls LLMEngine.step() continuously

3. LLMEngine.step() (vllm/engine/llm_engine.py)
   ├── 3a. Scheduler._schedule()
   │   ├── Check waiting queue (new requests)
   │   ├── Check running queue (ongoing generation)
   │   ├── Allocate/free KV cache blocks via BlockManager
   │   ├── Handle preemption if out of memory
   │   └── Return SchedulerOutputs (what to execute)
   │
   ├── 3b. Executor.execute_model(scheduler_outputs)
   │   ├── Send work to Worker(s)
   │   ├── Worker → ModelRunner.execute_model()
   │   │   ├── Prepare input tensors (token IDs, positions, attention metadata)
   │   │   ├── Run model forward pass
   │   │   │   ├── Embedding lookup
   │   │   │   ├── For each transformer layer:
   │   │   │   │   ├── Self-attention with PagedAttention
   │   │   │   │   ├── Read KV cache for decode, write new KV cache
   │   │   │   │   └── FFN (MLP)
   │   │   │   └── Final LM head → logits
   │   │   └── Sampler: apply logits processing → sample next token
   │   └── Return SamplerOutput (sampled tokens)
   │
   └── 3c. Process outputs
       ├── Append sampled tokens to sequences
       ├── Check stopping criteria (max tokens, EOS, stop strings)
       ├── Update sequence status (running → finished)
       ├── Free KV cache blocks for finished sequences
       └── Return RequestOutput to caller

4. API Server receives output
   ├── Format as OpenAI-compatible response
   ├── If streaming: yield SSE events
   └── If not streaming: return complete response
"""
print(request_flow)

## 8. Key Data Structures

Understanding vLLM's core data structures is essential for navigating the codebase.

In [ ]:
# Key data structures and where they live

data_structures = """
=== Core Data Structures ===

1. Sequence (vllm/sequence.py)
   - Represents a single sequence being processed
   - Contains: token IDs, logical token blocks, status
   - Status: WAITING → RUNNING → FINISHED_* or SWAPPED

2. SequenceGroup (vllm/sequence.py)
   - A group of sequences from the same request
   - For beam search: one SequenceGroup contains multiple Sequences
   - For simple generation: one SequenceGroup has one Sequence

3. SequenceData (vllm/sequence.py)
   - The actual token data for a sequence
   - prompt_token_ids: list of input token IDs
   - output_token_ids: list of generated token IDs

4. SamplingParams (vllm/sampling_params.py)
   - User-specified sampling parameters
   - temperature, top_p, top_k, max_tokens, etc.

5. SchedulerOutputs (vllm/core/scheduler.py)
   - Output of the scheduler for one step
   - scheduled_seq_groups: which sequences to run
   - blocks_to_swap_in/out: memory management ops
   - blocks_to_copy: for CoW (copy-on-write)

6. RequestOutput (vllm/outputs.py)
   - Final output returned to the user
   - Contains CompletionOutput(s) with generated text
"""
print(data_structures)

In [ ]:
# Source code: Sequence class (simplified)
# File: vllm/sequence.py

sequence_code = '''
class SequenceStatus(enum.IntEnum):
    """Status of a sequence."""
    WAITING = 0        # In the waiting queue
    RUNNING = 1        # Currently being processed
    SWAPPED = 2        # Swapped out to CPU
    FINISHED_STOPPED = 3   # Hit a stop condition
    FINISHED_LENGTH_CAPPED = 4  # Hit max_tokens
    FINISHED_ABORTED = 5   # Request was aborted
    FINISHED_IGNORED = 6   # Request was ignored (e.g., prompt too long)


class Sequence:
    """Stores the data, status, and block information for a sequence."""
    
    def __init__(self, seq_id, prompt, prompt_token_ids, block_size, ...):
        self.seq_id = seq_id
        self.prompt = prompt
        self.data = SequenceData(prompt_token_ids)
        self.status = SequenceStatus.WAITING
        
        # Block information for KV cache
        self.logical_token_blocks = []  # Logical blocks for this sequence
        self._append_tokens_to_blocks(prompt_token_ids)
    
    def get_len(self) -> int:
        """Total length = prompt + generated tokens."""
        return self.data.get_len()
    
    def get_num_new_tokens(self) -> int:
        """Tokens to be computed in next step.
        For prefill: all prompt tokens
        For decode: 1 token"""
        if self.data.stage == SequenceStage.PREFILL:
            return self.data.get_prompt_len()
        return 1
    
    def is_finished(self) -> bool:
        return self.status in [
            SequenceStatus.FINISHED_STOPPED,
            SequenceStatus.FINISHED_LENGTH_CAPPED,
            SequenceStatus.FINISHED_ABORTED,
            SequenceStatus.FINISHED_IGNORED,
        ]


class SequenceGroup:
    """A group of sequences generated from the same prompt."""
    
    def __init__(self, request_id, seqs, sampling_params, arrival_time, ...):
        self.request_id = request_id
        self.seqs_dict = {seq.seq_id: seq for seq in seqs}
        self.sampling_params = sampling_params
        self.arrival_time = arrival_time  # For FCFS scheduling
'''
print(sequence_code)

## 9. How to Navigate the Codebase Effectively

Here are practical strategies for finding your way around vLLM's large codebase.

In [ ]:
navigation_tips = """
=== Codebase Navigation Strategies ===

1. START FROM ENTRY POINTS
   - For serving: vllm/entrypoints/openai/api_server.py
   - For Python API: vllm/entrypoints/llm.py
   - Follow the call chain: LLM → LLMEngine → Scheduler + Executor → Worker → ModelRunner

2. FOLLOW THE IMPORTS
   - vLLM uses explicit imports. If you see:
     from vllm.core.scheduler import Scheduler
   - You know exactly where to find the Scheduler class.

3. USE THE REGISTRY
   - Models: vllm/model_executor/models/registry.py maps model names to classes
   - Attention: vllm/attention/selector.py shows which backend is selected when
   - Quantization: vllm/model_executor/layers/quantization/__init__.py

4. LOOK AT __init__.py FILES
   - They often re-export the most important classes
   - Example: vllm/__init__.py exports LLM and SamplingParams

5. SEARCH BY FEATURE
   - KV Cache: vllm/core/block_manager*.py + vllm/worker/cache_engine.py
   - Scheduling: vllm/core/scheduler.py
   - Attention: vllm/attention/
   - Sampling: vllm/model_executor/layers/sampler.py
   - LoRA: vllm/lora/
   - Speculative decoding: vllm/spec_decode/
   - Distributed: vllm/distributed/

6. TESTS REVEAL USAGE PATTERNS
   - tests/ mirrors the source structure
   - Tests show how components are meant to be used
   - Example: tests/core/test_scheduler.py shows Scheduler usage

7. CONFIGURATION TELLS YOU WHAT'S ADJUSTABLE
   - vllm/config.py defines all configuration dataclasses
   - vllm/engine/arg_utils.py shows all CLI arguments
   - These are great starting points for understanding capabilities
"""
print(navigation_tips)

## 10. Important Files Quick Reference

In [ ]:
# Quick reference of the most important files to know

important_files = {
    "Must-Read Files (start here)": {
        "vllm/entrypoints/llm.py": "High-level Python API (LLM class)",
        "vllm/engine/llm_engine.py": "Core engine (LLMEngine)",
        "vllm/core/scheduler.py": "Request scheduling logic",
        "vllm/worker/model_runner.py": "Model execution on GPU",
        "vllm/config.py": "All configuration dataclasses",
    },
    "Memory Management": {
        "vllm/core/block_manager.py": "KV cache block allocation (v1)",
        "vllm/core/block_manager_v2.py": "KV cache block allocation (v2)",
        "vllm/worker/cache_engine.py": "Physical KV cache tensor management",
    },
    "Model Architecture": {
        "vllm/model_executor/models/llama.py": "LLaMA model (most referenced)",
        "vllm/model_executor/models/registry.py": "Model name → class mapping",
        "vllm/model_executor/model_loader/loader.py": "Model weight loading",
    },
    "Attention": {
        "vllm/attention/attention.py": "Unified attention interface",
        "vllm/attention/selector.py": "Backend auto-selection",
        "vllm/attention/backends/flash_attn.py": "FlashAttention backend",
    },
    "Sampling": {
        "vllm/model_executor/layers/sampler.py": "Token sampling logic",
        "vllm/sampling_params.py": "Sampling configuration",
    },
    "API Server": {
        "vllm/entrypoints/openai/api_server.py": "FastAPI server setup",
        "vllm/entrypoints/openai/serving_chat.py": "Chat completions endpoint",
        "vllm/entrypoints/openai/protocol.py": "Request/response schemas",
    },
}

for category, files in important_files.items():
    print(f"\n{'=' * 60}")
    print(f"  {category}")
    print(f"{'=' * 60}")
    for filepath, description in files.items():
        print(f"  {filepath}")
        print(f"    → {description}")

## 11. Understanding the Version Structure

vLLM has gone through significant architectural changes. Here are the key evolutionary steps to understand when reading the code:

In [ ]:
version_evolution = """
=== vLLM Architecture Evolution ===

v0.1.x - v0.2.x (2023):
  - Original PagedAttention implementation
  - Single-GPU focus
  - Basic scheduler with FCFS

v0.3.x (Late 2023):
  - Added tensor parallelism support
  - Introduced Ray-based distributed execution
  - Added LoRA support
  - AWQ/GPTQ quantization support

v0.4.x (Early 2024):
  - Speculative decoding
  - Chunked prefill
  - Improved attention backends (FlashInfer)
  - BlockManagerV2 for better prefix caching

v0.5.x+ (2024):
  - Multi-modal support (vision-language models)
  - Improved pipeline parallelism
  - Structured output / guided decoding
  - Performance optimizations

Why This Matters:
  - You may see both v1 and v2 implementations of some components
  - block_manager.py vs block_manager_v2.py
  - Legacy code paths may still exist
  - Always check which version/path is currently active
"""
print(version_evolution)

## 12. Executor Architecture

The Executor is the bridge between the engine and the workers. Different executors support different deployment modes.

In [ ]:
executor_info = """
=== Executor Architecture ===

vllm/executor/
├── executor_base.py        # Abstract ExecutorBase
├── gpu_executor.py         # Single-node GPU executor
├── distributed_gpu_executor.py  # Multi-GPU executor base
├── ray_gpu_executor.py     # Ray-based distributed executor
├── multiproc_gpu_executor.py   # Multiprocessing-based executor
├── cpu_executor.py         # CPU-only executor
├── neuron_executor.py      # AWS Neuron executor
└── tpu_executor.py         # TPU executor

Selection Logic (simplified):
  if device == 'cpu':
      use CpuExecutor
  elif device == 'neuron':
      use NeuronExecutor
  elif tensor_parallel_size == 1 and pipeline_parallel_size == 1:
      use GPUExecutor  # Single GPU, simplest path
  elif use_ray:
      use RayGPUExecutor  # Ray for multi-node
  else:
      use MultiprocessingGPUExecutor  # Multi-GPU single node

The Executor's main job:
  1. Create and manage Worker instances
  2. Distribute work to workers
  3. Collect results from workers
  4. Handle initialization (model loading, KV cache allocation)
"""
print(executor_info)

## 13. Testing and Benchmarking

In [ ]:
testing_info = """
=== Tests Structure ===

tests/
├── core/
│   ├── test_scheduler.py        # Scheduler unit tests
│   ├── test_block_manager.py    # Block manager tests
│   └── ...
├── engine/
│   ├── test_llm_engine.py       # Engine tests
│   └── ...
├── models/
│   ├── test_models.py           # Model correctness tests
│   └── ...
├── worker/
│   ├── test_model_runner.py     # ModelRunner tests
│   └── ...
├── entrypoints/
│   ├── test_openai_server.py    # API endpoint tests
│   └── ...
├── samplers/
│   └── test_sampler.py          # Sampling tests
├── lora/
│   └── test_lora.py             # LoRA tests
├── spec_decode/
│   └── test_spec_decode.py      # Speculative decoding tests
└── ...

=== Benchmarks Structure ===

benchmarks/
├── benchmark_serving.py         # Online serving throughput benchmark
├── benchmark_throughput.py      # Offline throughput benchmark
├── benchmark_latency.py         # Latency benchmark
└── ...

Running Tests:
  pytest tests/core/test_scheduler.py -v
  pytest tests/ -k "test_llm_engine" -v

Running Benchmarks:
  python benchmarks/benchmark_throughput.py \\
    --model meta-llama/Llama-3-8B \\
    --input-len 512 --output-len 128 \\
    --num-prompts 100
"""
print(testing_info)

## 14. Common Code Patterns in vLLM

Recognizing these patterns will help you read the code faster.

In [ ]:
code_patterns = '''
=== Common Code Patterns ===

1. FACTORY PATTERN (from_engine_args)
   # Many classes have a class method that creates instances from EngineArgs
   engine = LLMEngine.from_engine_args(engine_args)
   # Internally: creates config objects, then constructs the engine

2. ABSTRACT BASE + CONCRETE IMPLEMENTATIONS
   # Used for: Executors, Workers, Attention Backends, Block Allocators
   class AttentionBackend(ABC):
       @abstractmethod
       def forward(self, ...): ...
   
   class FlashAttentionBackend(AttentionBackend):
       def forward(self, ...): ...  # Uses flash_attn

3. METADATA CLASSES
   # vLLM passes metadata objects to avoid long parameter lists
   class AttentionMetadata:
       # Contains everything the attention kernel needs:
       # - sequence lengths, block tables, is_prompt flags, etc.

4. SEQUENCE STATUS STATE MACHINE
   # Sequences move through states:
   # WAITING → RUNNING → FINISHED_*
   #              ↕
   #           SWAPPED

5. TWO-PHASE EXECUTION
   # Almost everything in vLLM follows:
   # Phase 1: Plan/Schedule (CPU-side)
   # Phase 2: Execute (GPU-side)
   # This allows overlapping CPU planning with GPU execution

6. PROFILING-BASED INITIALIZATION
   # KV cache size is determined by profiling:
   # 1. Load model onto GPU
   # 2. Measure remaining free memory
   # 3. Allocate that memory as KV cache blocks
   # See: Worker.determine_num_available_blocks()
'''
print(code_patterns)

## 15. Debugging and Development Tips

In [ ]:
debug_tips = """
=== Debugging & Development Tips ===

1. ENVIRONMENT VARIABLES FOR DEBUGGING:
   VLLM_LOGGING_LEVEL=DEBUG        # Verbose logging
   CUDA_VISIBLE_DEVICES=0          # Limit to specific GPU
   VLLM_ATTENTION_BACKEND=XFORMERS # Force specific attention backend
   VLLM_USE_FLASHINFER_SAMPLER=0   # Disable FlashInfer sampler

2. ENFORCE_EAGER MODE:
   # Disables CUDA graphs, making debugging easier
   llm = LLM(model="...", enforce_eager=True)
   # CUDA graphs capture a static computation graph, which makes
   # debugging with breakpoints impossible. Always use enforce_eager
   # when developing/debugging.

3. IMPORTANT LOG MESSAGES:
   # During startup, look for:
   "# GPU blocks: X, # CPU blocks: Y"    # Memory allocation
   "Using attention backend: ..."          # Which backend was selected
   "Model loaded successfully"             # Weight loading complete

4. PROFILING:
   # Use torch.profiler for GPU profiling:
   with torch.profiler.profile(
       activities=[torch.profiler.ProfilerActivity.CUDA]
   ) as prof:
       llm.generate(["Hello"], SamplingParams(max_tokens=10))
   print(prof.key_averages().table())

5. READING THE CODE:
   # Start with LLMEngine.step() - it's the heartbeat of the system
   # Then follow into scheduler._schedule() and executor.execute_model()
   # This covers ~80% of the core logic
"""
print(debug_tips)

## 16. Relationship Summary Diagram

In [ ]:
# Text-based architecture overview

architecture = """
┌──────────────────────────────────────────────────────────────────────┐
│                         USER INTERFACES                            │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────────────┐  │
│  │  vllm serve   │  │ Python LLM() │  │ openai.api_server        │  │
│  │  (CLI)        │  │ (offline)     │  │ (HTTP server)            │  │
│  └──────┬───────┘  └──────┬───────┘  └────────────┬─────────────┘  │
└─────────┼─────────────────┼───────────────────────┼────────────────┘
          │                 │                        │
          ▼                 │                        ▼
┌─────────────────┐        │           ┌────────────────────────┐
│ AsyncLLMEngine  │◄───────┘           │  AsyncLLMEngine        │
│ (async wrapper)  │                    │  (async wrapper)        │
└────────┬────────┘                    └───────────┬────────────┘
         │                                         │
         ▼                                         ▼
┌────────────────────────────────────────────────────────────────┐
│                        LLMEngine                               │
│  ┌─────────────────┐         ┌──────────────────────────────┐  │
│  │    Scheduler     │         │       Executor               │  │
│  │  ┌────────────┐  │         │  ┌──────────┐ ┌──────────┐  │  │
│  │  │ BlockMgr   │  │  ─────► │  │ Worker 0 │ │ Worker 1 │  │  │
│  │  │ (memory)   │  │         │  │┌────────┐│ │┌────────┐│  │  │
│  │  └────────────┘  │         │  ││ModelRun││ ││ModelRun││  │  │
│  └─────────────────┘         │  │└────────┘│ │└────────┘│  │  │
│                               │  └──────────┘ └──────────┘  │  │
│                               └──────────────────────────────┘  │
└────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────┐
│                    GPU Resources                               │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────────┐  │
│  │ Model Weights │  │  KV Cache    │  │  Attention Backend   │  │
│  │ (parameters)  │  │  (blocks)    │  │  (FlashAttn/etc)     │  │
│  └──────────────┘  └──────────────┘  └──────────────────────┘  │
└────────────────────────────────────────────────────────────────┘
"""
print(architecture)

## Exercises

### Exercise 1: Find Specific Functionality

Without looking at the solutions, try to identify which file(s) you would look at for each of these tasks:

1. Adding a new model architecture to vLLM
2. Changing how requests are prioritized 
3. Adding a new quantization method
4. Modifying the OpenAI-compatible chat endpoint
5. Changing how KV cache memory is allocated
6. Adding a new attention backend
7. Modifying how model weights are loaded
8. Adding a new CLI argument

In [ ]:
# Exercise 1 Solutions

solutions = {
    "1. New model architecture": [
        "vllm/model_executor/models/<new_model>.py  # Implement the model",
        "vllm/model_executor/models/registry.py     # Register the model",
    ],
    "2. Request prioritization": [
        "vllm/core/scheduler.py                     # Modify _schedule() or policy",
        "vllm/core/policy.py                        # Add new scheduling policy",
    ],
    "3. New quantization method": [
        "vllm/model_executor/layers/quantization/<new_method>.py  # Implement",
        "vllm/model_executor/layers/quantization/__init__.py     # Register",
    ],
    "4. Modify chat endpoint": [
        "vllm/entrypoints/openai/serving_chat.py    # Chat completions logic",
        "vllm/entrypoints/openai/protocol.py        # Request/response schemas",
    ],
    "5. KV cache allocation": [
        "vllm/core/block_manager.py or block_manager_v2.py  # Allocation logic",
        "vllm/core/block/                           # Block allocator implementations",
        "vllm/worker/cache_engine.py                # Physical memory management",
    ],
    "6. New attention backend": [
        "vllm/attention/backends/<new_backend>.py   # Implement backend",
        "vllm/attention/selector.py                 # Register for selection",
    ],
    "7. Model weight loading": [
        "vllm/model_executor/model_loader/loader.py     # Loading logic",
        "vllm/model_executor/model_loader/weight_utils.py # Weight utilities",
    ],
    "8. New CLI argument": [
        "vllm/engine/arg_utils.py                   # EngineArgs dataclass",
        "vllm/config.py                             # Corresponding config class",
    ],
}

for task, files in solutions.items():
    print(f"\n{task}:")
    for f in files:
        print(f"  → {f}")

### Exercise 2: Trace the Initialization

Starting from `LLM(model="meta-llama/Llama-3-8B")`, list every major class that gets instantiated and in what order. Compare your answer with the solution below.

In [ ]:
# Exercise 2 Solution

init_trace = """
Initialization Trace: LLM(model="meta-llama/Llama-3-8B")
═══════════════════════════════════════════════════════════

1. LLM.__init__()  [vllm/entrypoints/llm.py]
   └── Creates EngineArgs from user parameters
   
2. LLMEngine.from_engine_args()  [vllm/engine/llm_engine.py]
   ├── EngineArgs.create_engine_config() creates:
   │   ├── ModelConfig
   │   ├── CacheConfig
   │   ├── ParallelConfig
   │   ├── SchedulerConfig
   │   └── DeviceConfig
   │
   └── LLMEngine.__init__():
       ├── Creates Tokenizer
       ├── Creates Scheduler (with BlockSpaceManager inside)
       └── Creates Executor (e.g., GPUExecutor)
           └── GPUExecutor.__init__():
               └── Creates Worker
                   ├── Worker.__init__():
                   │   ├── Creates ModelRunner
                   │   └── Creates CacheEngine (later)
                   │
                   ├── Worker.init_device():
                   │   └── Sets up CUDA device, distributed env
                   │
                   ├── Worker.load_model():
                   │   └── ModelRunner.load_model()
                   │       └── ModelLoader loads weights from disk/HF
                   │           └── Creates Model (e.g., LlamaForCausalLM)
                   │
                   ├── Worker.determine_num_available_blocks():
                   │   └── Profiles GPU memory to determine KV cache size
                   │
                   └── Worker.initialize_cache():
                       └── CacheEngine allocates KV cache tensors

Total time: ~30-120 seconds (model dependent)
Most time spent: loading model weights from disk
"""
print(init_trace)

### Exercise 3: Map the Config

For each configuration parameter below, identify which config class it belongs to and what it controls:

1. `gpu_memory_utilization=0.95`
2. `tensor_parallel_size=4`  
3. `max_num_seqs=512`
4. `block_size=32`
5. `enforce_eager=True`
6. `enable_chunked_prefill=True`

In [ ]:
# Exercise 3 Solution

config_mapping = """
1. gpu_memory_utilization=0.95
   Config: CacheConfig
   Effect: Uses 95% of GPU memory for KV cache (after model weights).
   Higher = more sequences can run concurrently, but risk of OOM.

2. tensor_parallel_size=4
   Config: ParallelConfig
   Effect: Splits model across 4 GPUs using tensor parallelism.
   Each GPU holds 1/4 of each layer's weights.

3. max_num_seqs=512
   Config: SchedulerConfig
   Effect: Max 512 sequences can be in a single batch.
   Higher = more throughput but more memory per step.

4. block_size=32
   Config: CacheConfig
   Effect: Each KV cache block holds 32 tokens.
   Larger blocks = less overhead but more internal fragmentation.

5. enforce_eager=True
   Config: ModelConfig
   Effect: Disables CUDA graph capture. Slower but allows dynamic shapes
   and easier debugging. Always use this when developing.

6. enable_chunked_prefill=True
   Config: SchedulerConfig
   Effect: Allows splitting long prompts across multiple steps.
   Prevents one long prompt from blocking all other requests.
"""
print(config_mapping)

## Summary

In this notebook, we covered:

1. **Repository structure**: The main directories and their purposes
2. **Build system**: How setup.py and CMake work together for CUDA extensions
3. **Entry points**: CLI (`vllm serve`), API server, and Python API (`LLM` class)
4. **Configuration**: EngineArgs creates ModelConfig, CacheConfig, ParallelConfig, SchedulerConfig
5. **Dependency graph**: How components relate — Engine → Scheduler + Executor → Worker → ModelRunner
6. **Key data structures**: Sequence, SequenceGroup, SamplingParams, SchedulerOutputs
7. **Navigation strategies**: Start from entry points, follow imports, use the registry

**Next**: In Chapter 3.2, we'll dive deep into `LLMEngine` and `AsyncLLMEngine` — the beating heart of vLLM.